# DATASET YOUTUBE GADGET

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

zip_path = 'SENTIMENT 3 VIDEO YOUTUBE.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('data_youtube')

os.listdir('data_youtube')

In [ ]:
os.listdir('data_youtube/SENTIMENT 3 VIDEO YOUTUBE')

In [ ]:
import pandas as pd

df = pd.read_csv('data_youtube/SENTIMENT 3 VIDEO YOUTUBE/DATA_KOMEN_FULL_3_VIDEO.csv')
df

In [ ]:
df.info()

In [ ]:
df2 = pd.read_csv('data_gadgetin.csv')
df2

In [ ]:
df2.info()

## AMBIL KOLOM PENTING

In [ ]:
df = pd.read_csv('data_youtube/SENTIMENT 3 VIDEO YOUTUBE/DATA_KOMEN_FULL_3_VIDEO.csv')

df = df[['Channel_Sumber', 'Username', 'Komentar', 'Tanggal']]
df.columns = ['channel', 'username', 'text', 'tanggal']

In [ ]:
df2.columns

In [ ]:
df2 = pd.read_csv('data_gadgetin.csv')

df2 = df2[['Author', 'Comment', 'Time']]
df2.columns = ['username', 'text', 'tanggal']

df2['channel'] = 'gadgetin'

df2 = df2[['channel', 'username', 'text', 'tanggal']]

In [ ]:
print(df.columns)
print(df2.columns)

In [ ]:
df = pd.concat([df, df2], ignore_index=True)

In [ ]:
print("Jumlah data:", len(df))
df

### VOLUME

In [ ]:
print("Jumlah data total:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

# jumlah per channel
print("Jumlah data per channel:")
print(df['channel'].value_counts())

## CEK MISSING VALUE & DUPLIKAT

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.duplicated(subset=['text']).sum()

In [ ]:
df = df.drop_duplicates(subset=['text'])

In [ ]:
df.duplicated(subset=['text']).sum()

In [ ]:
print("Jumlah data setelah hapus duplikat:", len(df))

In [ ]:
df['text'] = df['text'].str.strip()

df = df.drop_duplicates(subset=['text'])

## CLEANING TEXT

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()  # lowercase

    text = re.sub(r'http\S+|www\S+', '', text)  # hapus URL
    text = re.sub(r'@\w+', '', text)            # hapus mention
    text = re.sub(r'#\w+', '', text)            # hapus hashtag
    text = re.sub(r'\d+', '', text)             # hapus angka
    text = re.sub(r'[^a-zA-Z\s]', '', text)     # hapus simbol & tanda baca
    text = re.sub(r'\s+', ' ', text).strip()    # hapus spasi berlebih

    return text

df['text'] = df['text'].apply(clean_text)

In [ ]:
df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')

In [ ]:
print("Jumlah data akhir:", len(df))
print("Duplikat:", df.duplicated().sum())
print("Missing value:\n", df.isnull().sum())

In [ ]:
df['panjang_komentar'] = df['text'].apply(len)
df['panjang_komentar'].describe()

In [ ]:
df['channel'].value_counts()

In [ ]:
df['tanggal'].dt.date.value_counts().sort_index()

### VELOCITY

In [ ]:
komentar_per_hari = df.groupby(df['tanggal'].dt.date).size()

print(komentar_per_hari.head())

komentar_per_hari.plot(figsize=(12,5))
plt.title("Pertumbuhan Komentar YouTube")
plt.xlabel("Tanggal")
plt.ylabel("Jumlah Komentar")
plt.show()

In [ ]:
from collections import Counter

all_text = ' '.join(df['text'])
kata = Counter(all_text.split())

top_kata = kata.most_common(10)

top_kata

##  SENTIMENT ANALYSIS

### METODE LEXICON

In [ ]:
positive_words = ['bagus','mantap','keren','love','suka','recommended','top','puas','worth']
negative_words = ['jelek','buruk','benci','gak suka','parah','kecewa','minus','lemot','mahal']

def sentiment_label(text):
    text = text.lower()

    score = 0

    for word in positive_words:
        if word in text:
            score += 1

    for word in negative_words:
        if word in text:
            score -= 1

    if score > 0:
        return 'positif'
    elif score < 0:
        return 'negatif'
    else:
        return 'netral'

df['sentiment'] = df['text'].apply(sentiment_label)

In [ ]:
df['sentiment'].value_counts()

## VISUALISASI

In [ ]:
import matplotlib.pyplot as plt

df['sentiment'].value_counts().plot(kind='bar')

plt.title('Distribusi Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)

plt.show()

In [ ]:
df.groupby('channel')['sentiment'].value_counts()

In [ ]:
df.groupby('channel')['sentiment'].value_counts().unstack().plot(kind='bar')

plt.title('Sentiment per Channel')
plt.xticks(rotation=45)
plt.ylabel('Jumlah')
plt.show()

In [ ]:
df['sentiment'].value_counts().plot(kind='pie', autopct='%1.1f%%')

plt.title('Persentase Sentiment')
plt.ylabel('')
plt.show()

In [ ]:
data_tanggal = df['tanggal'].dt.date.value_counts().sort_index()

data_tanggal.plot(figsize=(12,5))

plt.xticks(rotation=45)
plt.title('Jumlah Komentar per Hari')
plt.show()

In [ ]:
from wordcloud import WordCloud

text_all = ' '.join(df['text'])

wc = WordCloud(width=800, height=400).generate(text_all)

plt.imshow(wc)
plt.axis('off')
plt.show()

In [ ]:
df.to_csv('dataset sentiment_final.csv', index=False)
print("File berhasil disimpan!")

In [ ]:
df

## SENTIMENT ANALYSISI 2

### METODE INDOBERT

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [ ]:
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    label = torch.argmax(probs).item()

    if label == 0:
        return "negatif"
    elif label == 1:
        return "netral"
    else:
        return "positif"

In [ ]:
df_sample = df.sample(200)
df_sample['sentiment'] = df_sample['text'].apply(predict_sentiment)
df_sample

In [ ]:
df['sentiment'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

df['sentiment'].value_counts().plot(kind='bar')

plt.title('Distribusi Sentiment (IndoBERT)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
df_kata.head()

In [ ]:
df['panjang_komentar'].mean()

In [ ]:
df.groupby('channel')['sentiment'].value_counts()

In [ ]:
df.sample(10)

## PERBANDINGAN

### RULE-BASED

In [ ]:
positive_words = ['bagus','mantap','keren','love','suka','recommended','top','puas','worth']
negative_words = ['jelek','buruk','benci','gak suka','parah','kecewa','minus','lemot','mahal']

def sentiment_label(text):
    text = text.lower()
    score = 0

    for word in positive_words:
        if word in text:
            score += 1

    for word in negative_words:
        if word in text:
            score -= 1

    if score > 0:
        return 'positif'
    elif score < 0:
        return 'negatif'
    else:
        return 'netral'

df['sentiment_rule'] = df['text'].apply(sentiment_label)

### INDO BERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    label = torch.argmax(probs).item()

    if label == 0:
        return "negatif"
    elif label == 1:
        return "netral"
    else:
        return "positif"

df['sentiment_indo'] = df['text'].apply(sentiment_label)

In [ ]:
df_sample = df.sample(200)

df_sample['sentiment_indo'] = df_sample['text'].apply(predict_sentiment)

In [ ]:
df.columns

In [ ]:
df[['text', 'sentiment_rule', 'sentiment_indo']].head(50)

In [ ]:
df = df.sample(300)